In [31]:
from pathlib import Path
import pandas as pd
import os
from os import listdir
import random
import datetime
from os import listdir
from os.path import join as jp

import numpy as np
import pandas as pd
from pandas import DataFrame
from pandas import Series
import matplotlib.pyplot as plt
from math import radians, degrees, sin, cos, asin, acos, sqrt
import haversine
# import tensorflow

# from keras.models import Sequential
# from keras.layers import Dense, Input, Activation, BatchNormalization, Dropout

from sklearn import metrics
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

pd.options.mode.chained_assignment = None

In [2]:
wdir = os.path.join(os.getcwd(),'cabspottingdata')
file_names = [f for f in listdir(wdir)]
# file_names = file_names[:200]
data = pd.DataFrame()
# # file_names=['new_abboip.txt', 'new_abcoij.txt']
for file in file_names:
    temp = pd.read_csv(os.path.join(wdir, file), sep=" ", header=None)
    temp['cab_name'] = file.split('.')[0]
    data=data.append(temp, ignore_index=True)
data.columns = ['latitude', 'longitude', 'occupancy', 'time', 'cab_name']
data.head()

,latitude,longitude,occupancy,time,cab_name
0,37.75153,-122.39447,0,1211033530,new_abboip
1,37.75149,-122.39447,0,1211033614,new_abboip
2,37.75149,-122.39447,0,1211033674,new_abboip
3,37.75149,-122.39446,0,1211033735,new_abboip
4,37.75144,-122.39449,0,1211035303,new_abboip


In [3]:
data['cab_name'].nunique()

537

In [4]:
data.describe()

,latitude,longitude,occupancy,time
count,1.122006e+07,1.122006e+07,1.122006e+07,1.122006e+07
mean,3.776360e+01,-1.224124e+02,4.472238e-01,1.212036e+09
std,5.385999e-02,3.578201e-02,4.972069e-01,5.885683e+05
min,3.286970e+01,-1.270814e+02,0.000000e+00,1.211018e+09
25%,3.775513e+01,-1.224253e+02,0.000000e+00,1.211523e+09
50%,3.778106e+01,-1.224111e+02,0.000000e+00,1.212043e+09
75%,3.779045e+01,-1.224003e+02,1.000000e+00,1.212549e+09
max,5.030546e+01,-1.155622e+02,1.000000e+00,1.213090e+09


In [5]:
data.shape

(11220058, 5)

In [51]:
# from pandas_profiling import ProfileReport
# # EDA analysis
# profile = ProfileReport(data, title='Cab spotting Report', explorative=True)
# profile.to_widgets()
# profile.to_file("CabSpottingEDA.html")

Summarize dataset:   0%|          | 0/18 [00:00<?, ?it/s]

C:\Users\svi02\Anaconda3\lib\site-packages\scipy\stats\stats.py:4594: RuntimeWarning: overflow encountered in longlong_scalars
  (2 * xtie * ytie) / m + x0 * y0 / (9 * m * (size - 2)))


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render widgets:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

In [18]:
data.groupby(by='cab_name').count().sort_values("occupancy", ascending=True)

,latitude,longitude,occupancy,time
cab_name,,,,
new_eotcue,59,59,59,59
new_mfeuer,103,103,103,103
new_egoiwroi,185,185,185,185
new_oilrag,932,932,932,932
new_ifeshce,1003,1003,1003,1003
...,...,...,...,...
new_ejshigib,29536,29536,29536,29536
new_enkkand,29742,29742,29742,29742
new_equioc,29997,29997,29997,29997


In [13]:
import datetime as dt
dt.datetime.fromtimestamp(data.loc[data['cab_name'] =='new_eotcue', ['time']].values[0][0]).strftime('%Y-%m-%d %H:%M:%S')

'2008-05-22 16:33:48'

In [15]:
# data['timestamp']=
dt.datetime.fromtimestamp(data.loc[data['cab_name'] =='new_eotcue', ['time']]).strftime('%Y-%m-%d %H:%M:%S')

TypeError: an integer is required (got type DataFrame)

In [23]:
data['timestamp']=data['time'].apply(lambda x: dt.datetime.fromtimestamp(x).strftime('%Y-%m-%d %H:%M:%S'))
data.loc[data['cab_name'] =='new_eotcue']

,latitude,longitude,occupancy,time,cab_name,timestamp
4216711,37.75010,-122.39377,0,1211466828,new_eotcue,2008-05-22 16:33:48
4216712,37.75010,-122.39377,0,1211490788,new_eotcue,2008-05-22 23:13:08
4216713,37.75039,-122.39344,0,1212292269,new_eotcue,2008-06-01 05:51:09
4216714,37.75147,-122.39534,0,1212292331,new_eotcue,2008-06-01 05:52:11
4216715,37.75129,-122.39537,0,1212292381,new_eotcue,2008-06-01 05:53:01
4216716,37.75143,-122.39533,0,1212292441,new_eotcue,2008-06-01 05:54:01
4216717,37.75142,-122.39533,0,1212292501,new_eotcue,2008-06-01 05:55:01
4216718,37.75140,-122.39530,0,1212292566,new_eotcue,2008-06-01 05:56:06
4216719,37.75140,-122.39530,0,1212292621,new_eotcue,2008-06-01 05:57:01
4216720,37.74983,-122.39630,0,1212292681,new_eotcue,2008-06-01 05:58:01


In [32]:
import haversine

def PreviousCoordinates(df):
    df['prevlatitude'] = df.shift(1)['latitude']
    df['prevlongitude'] = df.shift(1)['longitude']
    df = df.dropna()
    return df

def EstimatedDistance(row):
    # formulation of finding distance via longitude and latitude in Miles from
    # https://gist.github.com/rochacbruno/2883505#gistcomment-1394026
    Longitude = row['longitude']
    Latitude = row['latitude']
    PrevLongitude = row['prevlongitude']
    PrevLatitude = row['prevlatitude']
    Longitude, Latitude, PrevLongitude, PrevLatitude = map(radians, 
                                                                    [Longitude, Latitude, 
                                                                    PrevLongitude, PrevLatitude])
    try:
        return 6371  * (acos(sin(Latitude) * sin(PrevLatitude) + cos(Latitude) * cos(PrevLatitude) * 
                                cos(Longitude - PrevLongitude)))
    except:
        return 0.0

def DistanceCalculation(df):
    df['kms'] = 0.0
    df['kms'] = df.apply(lambda row: EstimatedDistance(row), axis=1)
    return df

data=PreviousCoordinates(df=data)
data=DistanceCalculation(df=data)
data.head()
# Data = pd.DataFrame()
# for f in SelectedFiles:
#     df = ConverttoDF(file_name=f)
#     df = PreviousCoordinates(df=df)
#     df = DistanceCalculation(df=df)
#     Data = pd.concat([Data, df])

,latitude,longitude,occupancy,time,cab_name,timestamp,prevlatitude,prevlongitude,kms
2,37.75149,-122.39447,0,1211033674,new_abboip,2008-05-17 16:14:34,37.75149,-122.39447,0.000000
3,37.75149,-122.39446,0,1211033735,new_abboip,2008-05-17 16:15:35,37.75149,-122.39447,0.000880
4,37.75144,-122.39449,0,1211035303,new_abboip,2008-05-17 16:41:43,37.75149,-122.39446,0.006154
5,37.75151,-122.39453,0,1211035374,new_abboip,2008-05-17 16:42:54,37.75144,-122.39449,0.008542
6,37.75137,-122.39502,0,1211035434,new_abboip,2008-05-17 16:43:54,37.75151,-122.39453,0.045807


In [39]:
from haversine import haversine, Unit

lat1 = 37.75149; lat2 = 37.75144; long1 = -122.39446; long2 = -122.39449
print(haversine((lat1, long1), (lat2, long2)) )

0.006153669480742152


In [34]:
!pip install haversine